# L26 · Descubrimiento causal

**S26 · Lectura: Cap. 29 · 40–55 min**

## La pregunta

¿Qué evidencia hace creíble un grafo descubierto?

## Qué harás

Usar PC como generador de hipótesis, no como verdad automática.

- Lee la idea breve.
- Ejecuta la demostración y mira el gráfico.
- Completa las celdas marcadas.
- Escribe qué significa el resultado y qué no permite afirmar.

**Producto:** Un skeleton, conjuntos separadores y una lista de aristas a contrastar.

> **Idea clave:** Descubrir estructura ayuda a preguntar; no reemplaza el conocimiento del dominio.

La solución está en `la ruta de rescate docente (no incluida)`. Intenta primero el notebook del estudiante.

In [ ]:
# Localiza lab_common.py (carpeta actual o una superior)
import sys, pathlib
for _p in [pathlib.Path.cwd(), *list(pathlib.Path.cwd().parents)[:3]]:
    if (_p / "lab_common.py").exists():
        sys.path.insert(0, str(_p)); break

## 0. Preparación

In [ ]:
from lab_common import *
import numpy as np, pandas as pd, itertools
from scipy import stats, optimize as sopt, linalg as slin
rng = setup()
REPS = R(15, 40)

## 1. Leer un DAG: puerta trasera y suficiencia (Definición 29.1)
Antes de descubrir un grafo, hay que saber **leerlo**. Tomamos el grafo del retorno a la educación (Card, 1995) con nodos que **coinciden con los nombres de las columnas** que se usarían en `dowhy` (en el Listado 29.1 los ids `X, Z, Y, U` no coinciden con `treatment="educ"`, `outcome="lwage"`: E10).

$U$ (habilidad, **no observada**) $\to$ `educ` y $\to$ `lwage`; `exper` $\to$ `educ`, `lwage`; `nearc4` $\to$ `educ`; `educ` $\to$ `lwage`.

In [ ]:
NODOS = ["educ", "lwage", "exper", "nearc4", "habilidad"]; ix = {n: i for i, n in enumerate(NODOS)}
def grafo(aristas):
    A = np.zeros((len(NODOS), len(NODOS)))
    for a, b in aristas: A[ix[a], ix[b]] = 1
    return A
G = grafo([("habilidad", "educ"), ("habilidad", "lwage"), ("exper", "educ"), ("exper", "lwage"), ("nearc4", "educ"), ("educ", "lwage")])
def descendientes(A, v):
    out, pila = set(), [v]
    while pila:
        u = pila.pop()
        for w in range(len(A)):
            if A[u, w] and w not in out: out.add(w); pila.append(w)
    return out
def puerta_trasera(A, x, y, conjunto):
    """¿`conjunto` cumple el criterio de puerta trasera para (x → y)? (1) sin descendientes de x; (2) bloquea los caminos que entran a x."""
    ix_, iy, Z = ix[x], ix[y], [ix[c] for c in conjunto]
    if set(Z) & descendientes(A, ix_): return False
    A2 = A.copy(); A2[ix_, :] = 0                       # quita las flechas que salen de x: quedan solo caminos de puerta trasera
    return dsep(A2, [ix_], [iy], Z)
filas = {}
for conj in ([], ["exper"], ["exper", "nearc4"], ["exper", "habilidad"]):
    filas[", ".join(conj) or "(vacío)"] = {"¿puerta trasera?": "sí" if puerta_trasera(G, "educ", "lwage", conj) else "no", "¿observable?": "sí" if "habilidad" not in conj else "NO"}
show(pd.DataFrame(filas).T.rename_axis("conjunto de ajuste"))

**Mira el resultado.** El único conjunto que cierra la puerta trasera necesita un nodo que no se observa. ¿Qué dice eso del descubrimiento causal?

**Respuesta:**

Que **el grafo no sustituye a los datos que faltan**: con `habilidad` no observada ningún conjunto de variables medidas identifica el efecto de `educ`. Un algoritmo de descubrimiento solo ve las variables que le das (suficiencia causal); no puede añadir la que falta. Por eso el efecto de la educación en Card se estima con un **instrumento** (`nearc4`), no ajustando.



## 2. PC: el esqueleto
El esqueleto empieza con el grafo completo y quita la arista $i-j$ si existe un conjunto $S$ de vecinos con $X_i\perp X_j\mid S$ (test de Fisher-z sobre la correlación parcial), probando conjuntos de tamaño $0,1,2,\dots$ ("niveles").

El libro describe el paso; la versión ingenua **actualiza las adyacencias en cuanto quita una arista**, y entonces el resultado depende del **orden de las columnas**. La versión «estable» (Colombo y Maathuis, 2014) fija los vecinos al **inicio del nivel** y aplica las eliminaciones al **final**.

In [ ]:
def p_fisher(C, n, i, j, S):
    idx = [i, j, *S]; P = np.linalg.pinv(C[np.ix_(idx, idx)]); r = float(np.clip(-P[0, 1] / np.sqrt(P[0, 0] * P[1, 1]), -0.999999, 0.999999))
    z = 0.5 * np.log((1 + r) / (1 - r)) * np.sqrt(max(n - len(S) - 3, 1)); return float(2 * (1 - stats.norm.cdf(abs(z))))

def pc_esqueleto_inestable(X, alpha=0.05):
    """Versión ingenua: las aristas se quitan de inmediato y los vecinos se leen del grafo vivo."""
    n, d = X.shape; C = np.corrcoef(X.T); adj = ~np.eye(d, dtype=bool); sep = {}; nivel = 0
    while max(adj.sum(1)) - 1 >= nivel:
        for i in range(d):
            for j in range(i + 1, d):
                if not adj[i, j]: continue
                for ref in (i, j):
                    otros = [k for k in range(d) if adj[ref, k] and k not in (i, j)]
                    for S in itertools.combinations(otros, nivel):
                        if p_fisher(C, n, i, j, S) > alpha:
                            adj[i, j] = adj[j, i] = False; sep[(i, j)] = sep[(j, i)] = set(S); break
                    if not adj[i, j]: break
        nivel += 1
    return adj, sep

## 3. Completa tú: el esqueleto estable
Completa la parte marcada: al terminar de recorrer todos los pares de un nivel, **aplica** las eliminaciones guardadas (y registra los conjuntos separadores) y pasa al siguiente nivel. Los vecinos que se usan en el nivel salen de la **foto** tomada al inicio.

In [ ]:
# ✏️ Completa tú — pc_esqueleto_estable
# Pista: Para cada (i, j, S) en `quitar`: adj[i, j] = adj[j, i] = False y sep[(i, j)] = sep[(j, i)] = set(S). Después, nivel += 1.
def pc_esqueleto_estable(X, alpha=0.05):
    n, d = X.shape; C = np.corrcoef(X.T); adj = ~np.eye(d, dtype=bool); sep = {}; nivel = 0
    while max(adj.sum(1)) - 1 >= nivel:
        vecinos = {i: [k for k in range(d) if adj[i, k]] for i in range(d)}        # foto al INICIO del nivel
        quitar = []
        for i in range(d):
            for j in range(i + 1, d):
                if not adj[i, j]: continue
                for ref in (i, j):
                    otros = [k for k in vecinos[ref] if k not in (i, j)]
                    hallada = None
                    for S in itertools.combinations(otros, nivel):
                        if p_fisher(C, n, i, j, S) > alpha: hallada = S; break
                    if hallada is not None: quitar.append((i, j, hallada)); break
        # COMPLETA: aplica las eliminaciones de `quitar` (adj y sep, en las dos direcciones) y sube de nivel
        raise NotImplementedError("completa el final del nivel")
    return adj, sep

In [ ]:
# una cadena X0 → X1 → X2 y un colisionador X3 → X2 ← X4 (comprobación pequeña)
g = np.random.default_rng(0); e = g.normal(size=(2000, 5)); x = np.zeros((2000, 5)); x[:, 0] = e[:, 0]; x[:, 1] = x[:, 0] + e[:, 1]; x[:, 3] = e[:, 3]; x[:, 4] = e[:, 4]; x[:, 2] = x[:, 1] + x[:, 3] - x[:, 4] + e[:, 2]
adj_t, sep_t = pc_esqueleto_estable(x)
aristas_t = sorted((i, j) for i in range(5) for j in range(i + 1, 5) if adj_t[i, j])
lab_check("pc_esqueleto_estable", lambda: (aristas_t == [(0, 1), (1, 2), (2, 3), (2, 4)], sep_t.get((0, 2)) == {1}, (1, 3) in sep_t), [((), (True, True, True), 0)])

### ¿De verdad depende del orden la versión ingenua?
Generamos DAGs aleatorios (12 nodos, 16 aristas, n = 400: grafos algo densos y muestra moderada, donde los tests fallan más), **permutamos las columnas** y comparamos el esqueleto recuperado (reetiquetado a los nodos originales).

In [ ]:
def dag_aleatorio(d, n_aristas, g):
    orden = g.permutation(d); pares = [(orden[i], orden[j]) for i in range(d) for j in range(i + 1, d)]
    W = np.zeros((d, d))
    for k in g.choice(len(pares), n_aristas, replace=False): a, b = pares[k]; W[a, b] = g.choice([-1, 1]) * g.uniform(.5, 1.5)
    return W
def simular(W, n, g): return g.normal(size=(n, len(W))) @ np.linalg.inv(np.eye(len(W)) - W)
def esq_permutado(f, X, perm):
    adj, _ = f(X[:, perm]); inv = np.argsort(perm); return adj[np.ix_(inv, inv)]
dif = {"inestable": 0, "estable": 0}
for s in range(REPS):
    g = np.random.default_rng(100 + s); W = dag_aleatorio(12, 16, g); X = simular(W, 400, g); perm = g.permutation(12)
    for nom, f in (("inestable", pc_esqueleto_inestable), ("estable", pc_esqueleto_estable)):
        dif[nom] += int((f(X)[0] != esq_permutado(f, X, perm)).any())
show(pd.DataFrame({"esqueleto distinto al permutar": dif, "DAGs": {k: REPS for k in dif}}))

**Mira el resultado.** ¿Qué dice esto sobre comparar «un resultado de PC» entre dos corridas o dos bibliotecas?

**Respuesta:**

Que dos corridas con las mismas variables en otro orden pueden dar grafos distintos con la versión ingenua (en la corrida completa, `LAB_FULL=1`, ronda un tercio de los DAGs de este tamaño): la discrepancia es un artefacto del procesamiento, no de los datos. La versión estable evita **ese** problema (el esqueleto no depende del orden). Ojo: no evita los errores de los tests de independencia con n finito, ni la dependencia del orden en la orientación de colisionadores.



## 4. Orientar: colisionadores y reglas de Meek
Paso 2: para cada tripleta $i - k - j$ con $i,j$ no adyacentes, si $k\notin \text{sep}(i,j)$ se orienta $i\to k\leftarrow j$. Paso 3: las reglas de Meek propagan sin crear ciclos ni colisionadores nuevos. El resultado es una **CPDAG**: aristas que quedan sin orientar son las que los datos no pueden distinguir (clase de equivalencia de Markov).

In [ ]:
def orientar(adj, sep):
    d = len(adj); D = np.zeros((d, d), bool); U = adj.copy()
    for k in range(d):
        for i in range(d):
            for j in range(i + 1, d):
                if adj[i, k] and adj[j, k] and not adj[i, j] and k not in sep.get((i, j), set()):
                    for a in (i, j):
                        if U[a, k]: D[a, k] = True; U[a, k] = U[k, a] = False
    cambio = True
    while cambio:
        cambio = False
        for b in range(d):
            for c in range(d):
                if not U[b, c]: continue
                # R1: a→b, b—c, a y c no adyacentes ⇒ b→c
                if any(D[a, b] and not adj[a, c] and a != c for a in range(d)): D[b, c] = True; U[b, c] = U[c, b] = False; cambio = True; continue
                # R2: b→x→c con b—c ⇒ b→c
                if any(D[b, x] and D[x, c] for x in range(d)): D[b, c] = True; U[b, c] = U[c, b] = False; cambio = True; continue
                # R3: b—x, b—y, x→c, y→c, x e y no adyacentes ⇒ b→c
                xs = [x for x in range(d) if U[b, x] and D[x, c]]
                if any(not adj[x, y] for x, y in itertools.combinations(xs, 2)): D[b, c] = True; U[b, c] = U[c, b] = False; cambio = True
    return D, U
def pc(X, alpha=0.05):
    adj, sep = pc_esqueleto_estable(X, alpha); return orientar(adj, sep)
D_t, U_t = orientar(adj_t, sep_t)
print("aristas orientadas:", sorted((int(i), int(j)) for i, j in zip(*np.where(D_t))), "· sin orientar:", sorted((int(i), int(j)) for i, j in zip(*np.where(np.triu(U_t)))))

**Mira el resultado.** En el ejemplo, X0 → X1 → X2 ← X3: ¿por qué X0 — X1 queda sin orientar?

**Respuesta:**

Porque $X_0\to X_1$ y $X_0\leftarrow X_1$ son **observacionalmente equivalentes** dado el resto: ambos grafos tienen el mismo esqueleto y los mismos colisionadores. Solo el colisionador $X_1\to X_2\leftarrow X_3$ se identifica. Es el límite duro de la Jerarquía Causal visto desde la estructura (Cap. 3): ningún algoritmo que mire solo $P(v)$ puede separarlos.



## 5. Métricas: SHD, precisión y recall — y la Tabla 29.1 (E9)
Definimos con claridad: **esqueleto** (aristas sin dirección) y **dirigidas** (la orientación debe coincidir); una arista sin orientar en la CPDAG frente a una dirigida en la verdad cuenta como 1 en el SHD.

In [ ]:
def metricas(D, U, A_true):
    d = len(A_true); T = A_true != 0; Et = T | T.T; Ee = D | D.T | U; iu = np.triu_indices(d, 1)
    n_est, n_true, tp = int(Ee[iu].sum()), int(Et[iu].sum()), int((Ee & Et)[iu].sum())
    shd = 0
    for i, j in zip(*iu):
        if Ee[i, j] != Et[i, j]: shd += 1
        elif Ee[i, j]:
            ok = (D[i, j] and T[i, j]) or (D[j, i] and T[j, i])
            shd += 0 if ok else 1
    tp_dir = int((D & T).sum())
    return {"SHD": shd, "precisión (esqueleto)": tp / max(n_est, 1), "recall (esqueleto)": tp / max(n_true, 1),
            "precisión (dirigidas)": tp_dir / max(int(D.sum()), 1), "recall (dirigidas)": tp_dir / max(int(T.sum()), 1), "aristas estimadas": n_est, "aristas verdaderas": n_true}
# DGP del Listado 29.2: X0 -> X1 -> X2 <- X3, X4 aislada
g = np.random.default_rng(42); n_, d_ = 1000, 5
W_true = np.zeros((d_, d_)); W_true[0, 1] = 2.0; W_true[1, 2] = 1.5; W_true[3, 2] = 1.0
X_lst = g.standard_normal((n_, d_)) @ np.linalg.inv(np.eye(d_) - W_true)
D_l, U_l = pc(X_lst)
m_pc = metricas(D_l, U_l, W_true)
print("PC (α = 0,05) sobre el DGP del Listado 29.2:", {k: (round(v, 2) if isinstance(v, float) else v) for k, v in m_pc.items()})
print("aristas verdaderas del listado:", int((W_true != 0).sum()), "→ con 3 aristas el recall solo puede ser 0, 1/3, 2/3 o 1; el 0,75 de la Tabla 29.1 exige 4 aristas verdaderas")

**Mira el resultado.** La Tabla 29.1 da recall 0,75 para PC y dice «X4 aislada». ¿Qué está mal?

**Respuesta:**

Con el DGP del listado hay **3** aristas verdaderas, así que el recall del esqueleto solo puede tomar valores en múltiplos de 1/3; 0,75 = 3/4 exige 4 aristas, y el pie de la Figura 29.1 sí incluye una cuarta (X4→X1) que el listado no simula. Es una inconsistencia entre el listado, la tabla y la figura (E9). Aquí la tabla se **recalcula ejecutando**: lo que sale de la ejecución manda sobre lo que está impreso.



## 6. NOTEARS y varsortability
NOTEARS (Zheng et al., 2018) resuelve $\min_W \tfrac{1}{2n}\|X-XW\|_F^2+\lambda\|W\|_1$ sujeto a $h(W)=\operatorname{tr}\,e^{W\circ W}-d=0$ (acíclico), con Lagrangiano aumentado. Aquí va una versión mínima propia (el paquete de referencia no está disponible).

In [ ]:
def notears_lineal(X, lam=0.05, w_umbral=0.3, max_iter=60, h_tol=1e-8, rho_max=1e16):
    n, d = X.shape
    adjw = lambda w: (w[:d * d] - w[d * d:]).reshape(d, d)
    hfun = lambda W: float(np.trace(slin.expm(W * W)) - d)
    def f(w, rho, alpha):
        W = adjw(w); R = X - X @ W; loss = 0.5 / n * (R ** 2).sum(); Gl = -1 / n * X.T @ R
        E = slin.expm(W * W); h = np.trace(E) - d; Gh = E.T * W * 2
        obj = loss + 0.5 * rho * h * h + alpha * h + lam * w.sum(); g_ = Gl + (rho * h + alpha) * Gh
        return obj, np.concatenate((g_ + lam, -g_ + lam), axis=None)
    w = np.zeros(2 * d * d); rho, alpha, h = 1.0, 0.0, np.inf
    cotas = [(0, 0) if i == j else (0, None) for _ in range(2) for i in range(d) for j in range(d)]
    for _ in range(max_iter):
        while rho < rho_max:
            sol = sopt.minimize(f, w, args=(rho, alpha), method="L-BFGS-B", jac=True, bounds=cotas); w_new = sol.x; h_new = hfun(adjw(w_new))
            if h_new > 0.25 * h: rho *= 10
            else: break
        w, h = w_new, h_new; alpha += rho * h
        if h <= h_tol or rho >= rho_max: break
    W = adjw(w); W[np.abs(W) < w_umbral] = 0; return W
def varsortability(X, W_true):
    """Fracción de pares ancestro→descendiente (aristas) en que la varianza marginal crece a lo largo del orden causal."""
    v = X.var(0); pares = [(i, j) for i, j in zip(*np.where(W_true != 0))]; return float(np.mean([v[i] < v[j] for i, j in pares])) if pares else np.nan
Z_lst = (X_lst - X_lst.mean(0)) / X_lst.std(0)
filas = {}
for nom, XX in (("escala original", X_lst), ("estandarizado", Z_lst)):
    W_e = notears_lineal(XX); Dn, Un = (W_e != 0), np.zeros_like(W_e, bool)
    m = metricas(Dn, Un, W_true); m["varsortability"] = varsortability(XX, W_true); filas[nom] = m
show(pd.DataFrame(filas).T[["SHD", "precisión (esqueleto)", "recall (esqueleto)", "recall (dirigidas)", "varsortability"]].round(2))
print("W estimado (original):\n", np.round(notears_lineal(X_lst), 2)); print("W estimado (estandarizado):\n", np.round(notears_lineal(Z_lst), 2))

**Mira el resultado.** ¿Cambia la orientación al estandarizar? ¿Qué explica el cambio?

**Respuesta:**

Sí, y mucho: en este DGP, con la escala original NOTEARS recupera las tres aristas con la orientación correcta (SHD 0, varsortability 1,0) y con los mismos datos estandarizados la varsortability baja y el método **invierte la orientación y añade una arista de más** (SHD > 0, recall de aristas dirigidas casi nulo). Es el fenómeno de Reisach et al. (2021): en la escala original la varianza crece a lo largo del orden causal y el método puede estar explotando esa regularidad más que la estructura. Un método que cambia de respuesta cuando solo cambió la unidad de medida no está describiendo la causalidad. PC trabaja con correlaciones y no cambia al estandarizar. Por eso el libro pide reportar ambas corridas.



## 7. Un benchmark con grafo conocido: `10_vars_10_edges`
Comparamos PC (estable) y NOTEARS (escala original y estandarizado) contra el grafo verdadero (`graph_adjmatrix.csv`). *Supuesto de lectura*: `adj[i, j] = 1` significa $i\to j$.

In [ ]:
sc = load_scenario("synthetic/linear_gaussian/10_vars_10_edges")
Xb = sc.train[sc.covariates].to_numpy(float); A_true = np.asarray(sc.adj, float)
Zb = (Xb - Xb.mean(0)) / Xb.std(0)
print(f"{Xb.shape[0]} filas de entrenamiento · {int((A_true != 0).sum())} aristas verdaderas · varsortability de los datos = {varsortability(Xb, A_true):.2f}")
filas = {}
Dp, Up = pc(Xb); filas["PC estable (α = 0,05)"] = metricas(Dp, Up, A_true)
for nom, XX in (("NOTEARS (original)", Xb), ("NOTEARS (estandarizado)", Zb)):
    filas[nom] = metricas(notears_lineal(XX) != 0, np.zeros_like(A_true, bool), A_true)
show(pd.DataFrame(filas).T[["SHD", "precisión (esqueleto)", "recall (esqueleto)", "precisión (dirigidas)", "recall (dirigidas)"]].round(2))

In [ ]:
# Vista rápida de los datos del escenario
import matplotlib.pyplot as plt
import numpy as np
_fig, _ax = plt.subplots(1, 2, figsize=(9, 3.2))
try:
    _scenario = globals().get("sc") or globals().get("lc")
    if _scenario is None:
        raise NameError("No hay un escenario cargado para el gráfico editorial")
    _df = _scenario.train
    if _scenario.treatment is not None and _scenario.outcome is not None:
        _t = _df[_scenario.treatment].to_numpy(float)
        _y = _df[_scenario.outcome].to_numpy(float)
        if np.unique(_t).size == 2:
            _counts = [int((_t == value).sum()) for value in (0, 1)]
            _ax[0].bar(["control", "tratado"], _counts)
            _ax[0].set_title("Asignación")
            _ax[1].boxplot([_y[_t == 0], _y[_t == 1]], labels=["control", "tratado"])
            _ax[1].set_title("Resultado observado")
        else:
            _ax[0].hist(_t, bins=20)
            _ax[0].set_title(_scenario.treatment)
            _ax[1].scatter(_t, _y, s=8, alpha=.35)
            _ax[1].set_title(f"{_scenario.treatment} frente a {_scenario.outcome}")
    elif _scenario.adj is not None:
        _ax[0].imshow(_scenario.adj, cmap="Greys", interpolation="nearest")
        _ax[0].set_title("Grafo del escenario")
        _ax[1].axis("off")
    else:
        _ax[0].axis("off"); _ax[1].axis("off")
    _fig.suptitle(f"Escenario: {_scenario.name}")
    _fig.tight_layout(); plt.show()
finally:
    plt.close(_fig)

**Mira el resultado.** ¿Qué método gana aquí, y qué dice la varsortability sobre cuánto confiar en eso?

**Respuesta:**

En la corrida con la escala original NOTEARS acierta casi todo, pero con los datos estandarizados es el peor de los tres: la ventaja desaparece cuando se quita la señal de escala (la varsortability de estos datos es alta, ver la línea de arriba). PC no depende de la escala y se queda en el medio: recupera casi todo el esqueleto y no orienta lo que los datos no distinguen. Es **una** muestra y **un** grafo: para ordenar métodos hay que repetir con varias semillas y varios grafos (ejercicio opcional). La conclusión robusta es otra: antes de dar crédito a una orientación de NOTEARS, mira si sobrevive a estandarizar.



## 8. Sin verdad: estabilidad por submuestreo y placebo de aristas
En datos reales no hay `graph_adjmatrix.csv`. Dos controles baratos: **(a)** repetir PC en submuestras y contar con qué **frecuencia** aparece cada arista (< 50 % es sospechosa); **(b)** añadir variables **placebo** independientes y comprobar que no se conectan.

In [ ]:
g = np.random.default_rng(5); d0 = Xb.shape[1]; N_PL = 3
Xp = np.c_[Xb, g.normal(size=(len(Xb), N_PL)) * Xb.std(0).mean()]                     # placebo: ruido independiente de todo
freq = np.zeros((Xp.shape[1],) * 2); n_sub = R(15, 40)
for s in range(n_sub):
    idx = np.random.default_rng(900 + s).choice(len(Xp), len(Xp) // 2, replace=False); adj_s, _ = pc_esqueleto_estable(Xp[idx]); freq += adj_s
freq /= n_sub
T = (A_true != 0) | (A_true != 0).T; iu = np.triu_indices(d0, 1)
f_ver = freq[:d0, :d0][iu][T[iu]]; f_fal = freq[:d0, :d0][iu][~T[iu]]
f_pl = freq[:d0, d0:].ravel()
show(pd.DataFrame({"aristas verdaderas": [len(f_ver), f_ver.mean(), (f_ver >= .5).mean()], "no aristas": [len(f_fal), f_fal.mean(), (f_fal >= .5).mean()],
                   "aristas hacia placebos": [len(f_pl), f_pl.mean(), (f_pl >= .5).mean()]}, index=["cuántos pares", "frecuencia media", "fracción con frecuencia ≥ 50 %"]).T.round(3))

**Mira el resultado.** ¿Separa la frecuencia las aristas verdaderas de las falsas? ¿Se conectan los placebos?

**Respuesta:**

Compara las tres filas. Si la frecuencia media de las verdaderas es alta y la de las falsas y los placebos baja, la regla «< 50 % es sospechosa» funciona **en este caso**; pero la estabilidad mide la consistencia del método ante el remuestreo, no la verdad: una arista espuria que depende de un confusor omitido puede aparecer estable en todas las submuestras. Los placebos son útiles porque tienen verdad conocida (ninguna arista): cualquier conexión es un falso positivo.



## 9. Señal de alerta: el grafo estimado no es evidencia, es una guía
Suficiencia causal: si falta una causa común, PC la ignora. Con `latent_confounder` (la variable `X0` es el confusor latente de `X8 → X9`, efecto verdadero −1,82) ajustamos por **todo lo que el grafo observado deja ajustar**.

In [ ]:
# EVALUATION-CELL (usa la verdad simulada)
sl = load_scenario("synthetic/edge_cases/latent_confounder")
obs = sl.train[sl.covariates + [sl.treatment, sl.outcome]]; Xo = obs.to_numpy(float)
Dl, Ul = pc(Xo); cols = list(obs.columns); iT, iY = cols.index(sl.treatment), cols.index(sl.outcome)
print("PC sobre las variables observadas: ¿hay arista", sl.treatment, "—", sl.outcome, "?", bool((Dl | Dl.T | Ul)[iT, iY]))
r_obs = ols(sl.train[sl.outcome].to_numpy(float), np.c_[sl.train[sl.treatment], sl.train[sl.covariates]])
r_full = ols(sl.full.iloc[:len(sl.train)][sl.outcome].to_numpy(float), np.c_[sl.full.iloc[:len(sl.train)][sl.treatment], sl.full.iloc[:len(sl.train)][sl.covariates + ["X0"]]]) if sl.full is not None else None
print(f"efecto ajustando por lo observado: {r_obs['coef'][1]:.3f} (EE {r_obs['se'][1]:.3f}) · verdad: {sl.true_ate:.3f}")
if r_full is not None: print(f"efecto ajustando también por el confusor latente X0: {r_full['coef'][1]:.3f}")

**Mira el resultado.** PC encuentra la arista X8 — X9 y aun así el efecto ajustado por lo observado falla. ¿Por qué no hay contradicción?

**Respuesta:**

Porque el grafo estimado es **correcto en lo que muestra** (X8 y X9 están conectadas) y a la vez **no contiene al confusor que importa**. La arista dice que hay relación, no cuánto de ella es causal. Con suficiencia causal rota, ajustar por lo que el grafo sugiere deja un sesgo. Úsese el grafo estimado como **hipótesis para revisar con un experto**, no como el DAG que justifica el ajuste.



## 10. Supuesto y bitácora

In [ ]:
fila_b = bitacora(
    pregunta="¿Qué estructura causal sugieren los datos y cuánto confiar en ella?",
    metodo="PC estable (Fisher-z, α = 0,05) con orientación de colisionadores y reglas de Meek; NOTEARS lineal con y sin estandarizar; estabilidad por submuestreo y placebos",
    estimando="El grafo (clase de equivalencia de Markov), no un efecto",
    numero_ic=f"10_vars_10_edges: PC SHD {filas['PC estable (α = 0,05)']['SHD']}, recall (esqueleto) {filas['PC estable (α = 0,05)']['recall (esqueleto)']:.2f}; NOTEARS original SHD {filas['NOTEARS (original)']['SHD']} frente a estandarizado {filas['NOTEARS (estandarizado)']['SHD']}",
    supuesto=supuesto("no observable", "declarado y acotado", "suficiencia causal y fidelidad: sin confusores omitidos ni cancelaciones exactas; con el grafo conocido solo en simulación y la varsortability como señal de que la escala puede estar ayudando"),
    decision="alertar", lab="L26")

---
## Qué queda para el libro
Quedan en el libro: GES, FCI/LiNGAM y el análisis de alta dimensión de Kalisch y Bühlmann (2007). `causal-learn` y el paquete `notears` del repositorio de Zheng et al. no están disponibles aquí: se usan implementaciones propias mínimas (PC de ~60 líneas y NOTEARS lineal de ~30). Si los tienes, compáralos con estas.

*Material formativo (sin nota). Los datos son sintéticos: ver la ficha del escenario.*